In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/mlfp04/ex_5.py ──
"""
Shared infrastructure for MLFP04 Exercise 5 — Association Rules.

Contains: synthetic Singapore retail transaction generator, category map,
rule dataclass helpers, and small polars utilities used by every technique
file in ``modules/mlfp04/solutions/ex_5/``.

Technique-specific code (Apriori from scratch, FP-Growth wrapper, rule
evaluation, feature engineering for classification) does NOT belong here —
it lives in the per-technique files.
"""

import asyncio
import re
from collections import defaultdict
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import polars as pl

from kailash_ml import ExperimentTracker


setup_environment()

# ════════════════════════════════════════════════════════════════════════
# OUTPUT DIRECTORY
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "mlfp04_ex5_association_rules"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# SINGAPORE RETAIL PRODUCT CATALOGUE
# ════════════════════════════════════════════════════════════════════════
# 25 products grouped to mirror a typical HDB neighbourhood mini-mart basket.

PRODUCTS: list[str] = [
    "bread",
    "butter",
    "milk",
    "eggs",
    "rice",
    "noodles",
    "soy_sauce",
    "cooking_oil",
    "chicken",
    "fish",
    "coffee",
    "tea",
    "sugar",
    "condensed_milk",
    "biscuits",
    "chips",
    "soft_drink",
    "beer",
    "wine",
    "tissue",
    "shampoo",
    "soap",
    "detergent",
    "toothpaste",
    "bananas",
]

CATEGORY_MAP: dict[str, str] = {
    "bread": "breakfast",
    "butter": "breakfast",
    "eggs": "breakfast",
    "milk": "dairy",
    "condensed_milk": "dairy",
    "coffee": "beverage",
    "tea": "beverage",
    "soft_drink": "beverage",
    "sugar": "pantry",
    "rice": "pantry",
    "cooking_oil": "pantry",
    "soy_sauce": "pantry",
    "noodles": "pantry",
    "chicken": "protein",
    "fish": "protein",
    "beer": "alcohol",
    "wine": "alcohol",
    "chips": "snack",
    "biscuits": "snack",
    "bananas": "fruit",
    "shampoo": "personal_care",
    "soap": "personal_care",
    "toothpaste": "personal_care",
    "tissue": "household",
    "detergent": "household",
}

# Co-purchase bundles (items, probability). Models real behaviour:
# kaya-toast breakfast, kopi-C, household replenishment, beer+chips, etc.
BUNDLES: list[tuple[list[str], float]] = [
    (["bread", "butter", "eggs"], 0.15),
    (["coffee", "condensed_milk", "sugar"], 0.12),
    (["rice", "chicken", "soy_sauce"], 0.10),
    (["noodles", "eggs", "soy_sauce"], 0.08),
    (["beer", "chips"], 0.09),
    (["milk", "biscuits"], 0.07),
    (["shampoo", "soap", "toothpaste"], 0.06),
    (["tea", "sugar", "biscuits"], 0.05),
    (["wine", "chips", "biscuits"], 0.04),
    (["cooking_oil", "rice", "fish"], 0.06),
    (["detergent", "tissue", "soap"], 0.05),
    (["bananas", "milk", "eggs"], 0.05),
]

N_TRANSACTIONS_DEFAULT: int = 2500


# ════════════════════════════════════════════════════════════════════════
# TRANSACTION GENERATOR
# ════════════════════════════════════════════════════════════════════════


def _draw_basket(rng: np.random.Generator, bundle_probs: Iterable[float]) -> set[str]:
    """Draw one basket: each bundle fires with its probability, each item in
    a firing bundle is kept with 0.85 probability, plus a Poisson(2) number
    of random impulse items."""
    basket: set[str] = set()
    for (bundle_items, _), prob in zip(BUNDLES, bundle_probs):
        if rng.random() < prob:
            for item in bundle_items:
                if rng.random() < 0.85:
                    basket.add(item)
    n_random = rng.poisson(2)
    if n_random > 0:
        random_items = rng.choice(PRODUCTS, size=int(min(n_random, 5)), replace=False)
        basket.update(random_items)
    return basket


def generate_transactions(
    n: int = N_TRANSACTIONS_DEFAULT,
    seed: int = 42,
) -> list[set[str]]:
    """Generate ``n`` non-empty synthetic Singapore retail transactions.

    Each transaction is a set of product strings. Bundles fire with their
    listed probability; each item inside a firing bundle is kept with 0.85
    probability (random drop-out) so support is noisy. A Poisson number of
    random items is added on top to simulate impulse buys. Empty draws are
    re-drawn, so exactly ``n`` baskets are returned.
    """
    rng = np.random.default_rng(seed)
    base_probs = [prob for _, prob in BUNDLES]
    transactions: list[set[str]] = []
    while len(transactions) < n:
        basket = _draw_basket(rng, base_probs)
        if basket:
            transactions.append(basket)
    return transactions


# Habit model for two-trip shoppers (used by Exercise 5.4). A shopper is a
# "regular" for a bundle with probability HABIT_RATE_MULTIPLIER x the
# bundle's base rate; regulars fire that bundle on HABIT_FIRE_PROB of their
# trips, everyone else on CASUAL_FIRE_PROB of their trips.
HABIT_RATE_MULTIPLIER: float = 1.5
HABIT_FIRE_PROB: float = 0.6
CASUAL_FIRE_PROB: float = 0.03


def generate_shopper_trips(
    n_shoppers: int = N_TRANSACTIONS_DEFAULT,
    seed: int = 42,
) -> tuple[list[set[str]], list[set[str]]]:
    """Two consecutive baskets per shopper: ``(this_trip, next_trip)``.

    Both trips are drawn independently from the SAME per-shopper habit
    profile, so this trip carries real but noisy evidence about the next
    one. The next trip is never used to build features — it only supplies
    a prediction target that is NOT a function of this trip's items.
    """
    rng = np.random.default_rng(seed)
    base = np.array([prob for _, prob in BUNDLES])
    this_trip: list[set[str]] = []
    next_trip: list[set[str]] = []
    while len(this_trip) < n_shoppers:
        habit = rng.random(len(BUNDLES)) < base * HABIT_RATE_MULTIPLIER
        probs = np.where(habit, HABIT_FIRE_PROB, CASUAL_FIRE_PROB)
        first = _draw_basket(rng, probs)
        second = _draw_basket(rng, probs)
        if first:
            this_trip.append(first)
            next_trip.append(second)
    return this_trip, next_trip


def transactions_to_onehot(transactions: list[set[str]]) -> pl.DataFrame:
    """One-row-per-transaction boolean matrix (columns = sorted PRODUCTS).

    Polars-native. mlxtend needs pandas, so callers convert with
    ``.to_pandas()`` at the mlxtend call site only.
    """
    all_items = sorted(PRODUCTS)
    rows = [{item: (item in txn) for item in all_items} for txn in transactions]
    return pl.DataFrame(rows)


def product_frequency(transactions: Iterable[set[str]]) -> dict[str, int]:
    """Count how many transactions contain each product."""
    counts: dict[str, int] = defaultdict(int)
    for txn in transactions:
        for item in txn:
            counts[item] += 1
    return dict(counts)


def print_transaction_summary(transactions: list[set[str]]) -> None:
    """One-liner summary + top 10 product frequency. Used by every file."""
    avg_basket = float(np.mean([len(t) for t in transactions]))
    print("=== Synthetic Singapore Retail Transactions ===")
    print(f"  Transactions: {len(transactions):,}")
    print(f"  Products:     {len(PRODUCTS)}")
    print(f"  Avg basket:   {avg_basket:.1f} items")

    freq = product_frequency(transactions)
    n = len(transactions)
    print("\n  Top 10 products by frequency:")
    for item, count in sorted(freq.items(), key=lambda kv: -kv[1])[:10]:
        print(f"    {item:<20} {count:>5} ({count / n:.1%})")


# ════════════════════════════════════════════════════════════════════════
# RULE HELPERS
# ════════════════════════════════════════════════════════════════════════


def format_itemset(items: Iterable[str]) -> str:
    """Deterministic pretty-print of a frozenset of items."""
    return ", ".join(sorted(items))


def categorise_rule(
    antecedent: frozenset[str],
    consequent: frozenset[str],
) -> tuple[set[str], set[str], str]:
    """Return (antecedent_categories, consequent_categories, relation_type)."""
    ant_cats = {CATEGORY_MAP.get(item, "other") for item in antecedent}
    con_cats = {CATEGORY_MAP.get(item, "other") for item in consequent}
    if ant_cats == con_cats:
        rel = "within-category complement"
    elif ant_cats & con_cats:
        rel = "cross-category with overlap"
    else:
        rel = "cross-category association"
    return ant_cats, con_cats, rel


def rules_to_polars(rules: list[dict]) -> pl.DataFrame:
    """Convert a list of rule dicts into a polars DataFrame for plotting."""
    return pl.DataFrame(
        {
            "antecedent": [format_itemset(r["antecedent"]) for r in rules],
            "consequent": [format_itemset(r["consequent"]) for r in rules],
            "support": [float(r["support"]) for r in rules],
            "confidence": [float(r["confidence"]) for r in rules],
            "lift": [float(r["lift"]) for r in rules],
        }
    )


# ════════════════════════════════════════════════════════════════════════
# KAILASH-ML EXPERIMENT TRACKER — shared by every association-rules technique
# ════════════════════════════════════════════════════════════════════════
# Every M4 ex_5 lesson logs its mining + rule-evaluation + classifier runs
# into a single SQLite store so students can compare techniques side by side
# (Apriori vs FP-Growth runtime, rule-quality at varying thresholds,
# baseline vs rule-enhanced classifier metrics) after the lesson group ends.
# Mirrors m4_clustering_zoo / m4_dimreduction_zoo / m4_anomaly_zoo from
# ex_1 / ex_3 / ex_4.
#
# IMPORTANT: this store is SEPARATE from the other M4 stores. Per the
# SQLite contention trap (.session-notes), running two ExperimentTracker
# writers against the same SQLite file concurrently triggers `disk I/O
# error`. Each exercise group gets its own DB.

ASSOC_DB = "sqlite:///mlfp04_ex5_assoc_rules.db"
ASSOC_EXPERIMENT_NAME = "m4_assoc_rules_zoo"

_TRACKER_KEY_RE = re.compile(r"[^a-zA-Z0-9_.\-]")


def _slug(text: str) -> str:
    """Coerce free-form prose into a tracker-key-safe slug.

    ExperimentTracker enforces ``^[a-zA-Z_][a-zA-Z0-9_.\\-]*$`` on metric
    names. Replace forbidden characters with underscores; ensure a leading
    alpha character; collapse repeats.
    """
    s = _TRACKER_KEY_RE.sub("_", text.strip())
    s = re.sub(r"_+", "_", s).strip("_") or "metric"
    if not s[0].isalpha() and s[0] != "_":
        s = f"m_{s}"
    return s


def _finite(x: float) -> float:
    """Tracker rejects NaN/inf via MetricValueError; coerce to 0.0.

    Association-rule metrics (lift, conviction) can collapse to inf when
    confidence saturates at 1.0; aggregate scalars over empty filter sets
    can NaN. Every emit through this guard.
    """
    return float(x) if x == x and x not in (float("inf"), float("-inf")) else 0.0


async def _setup_engines_async() -> tuple[ExperimentTracker, str]:
    """Open the association-rules ExperimentTracker."""
    tracker = await ExperimentTracker.create(store_url=ASSOC_DB)
    return tracker, ASSOC_EXPERIMENT_NAME


def setup_engines() -> tuple[ExperimentTracker, str]:
    """Sync wrapper. Returns (tracker, experiment_name)."""
    return asyncio.run(_setup_engines_async())


def teardown_engines(tracker: ExperimentTracker) -> None:
    """Drain the aiosqlite worker threads before the script returns.

    kailash's AsyncSQLitePool spawns NON-DAEMON aiosqlite worker threads on
    first pool use. Python 3.13's ``Py_FinalizeEx`` joins non-daemon threads
    BEFORE running ``atexit`` handlers, so an atexit-based close runs too
    late — the interpreter hangs forever in ``wait_for_thread_shutdown``
    waiting on workers stuck in ``queue.get()``.

    Solutions MUST call ``teardown_engines(tracker)`` after the REFLECTION
    block (or use it via ``try: … finally: teardown_engines(tracker)``).
    See ``rules/patterns.md`` § "Async Resource Cleanup": real cleanup is
    the caller's responsibility; ``close()`` must NOT live in ``__del__``.
    """
    asyncio.run(tracker.close())


async def _track_run_async(
    tracker: ExperimentTracker,
    exp_name: str,
    run_name: str,
    params: dict[str, Any],
    scalar_metrics: dict[str, float],
    series_metrics: dict[str, list[float]] | None = None,
) -> None:
    """Log one lesson's run: scalar metrics + optional per-step series."""
    async with tracker.track(experiment=exp_name, run_name=run_name) as run:
        await run.log_params({k: str(v) for k, v in params.items()})
        for name, value in scalar_metrics.items():
            await run.log_metric(_slug(name), _finite(value))
        if series_metrics:
            for name, values in series_metrics.items():
                slug = _slug(name)
                for step, value in enumerate(values, start=1):
                    await run.log_metric(slug, _finite(value), step=step)


def track_run(
    tracker: ExperimentTracker,
    exp_name: str,
    run_name: str,
    params: dict[str, Any],
    scalar_metrics: dict[str, float],
    series_metrics: dict[str, list[float]] | None = None,
) -> None:
    """Sync wrapper for logging a single technique's run."""
    asyncio.run(
        _track_run_async(
            tracker, exp_name, run_name, params, scalar_metrics, series_metrics
        )
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 5.3: Rule Evaluation — Support, Confidence, Lift, Conviction
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Turn frequent itemsets into directional association rules (X -> Y)
#   - Compute the four standard rule-quality metrics
#   - Apply a three-threshold filter (support + confidence + lift)
#   - Separate cross-category rules from within-category rules
#   - Rank actionable rules for business stakeholders
#
# PREREQUISITES:
#   - 01_apriori_from_scratch.py (frequent itemset mining)
#   - Basic probability (conditional probability, independence)
#
# ESTIMATED TIME: ~40 min
#
# TASKS:
#   1. Theory — what each rule metric means mathematically and in business terms
#   2. Build — implement `generate_rules()` and the three-threshold filter
#   3. Train — mine + score rules on the Singapore retail basket
#   4. Visualise — top rules table + support/confidence/lift scatter
#   5. Apply — health & beauty retailer "buy-together" recommender
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

from collections import defaultdict
from itertools import combinations


# ── Kailash-ML ExperimentTracker — every association-rules run logs here ─
tracker, exp_name = setup_engines()



## THEORY — The Four Rule Quality Metrics


In [ ]:
# A rule "X -> Y" says: baskets containing X tend to also contain Y.
# It is a statistical summary, not a causal claim. We score it with four
# metrics because each one catches a different failure mode.
#
#   SUPPORT     supp(X u Y) = count(X u Y) / N
#               How often the pair appears. Low support = not enough data
#               to trust the rule; rules with support < 1% are noise on
#               most retail datasets.
#
#   CONFIDENCE  conf(X -> Y) = supp(X u Y) / supp(X)
#               P(Y given X). How RELIABLE the rule is when X is present.
#               A 95% confidence rule fires ~19 out of 20 times X appears.
#
#   LIFT        lift(X -> Y) = conf(X -> Y) / supp(Y)
#               = P(Y|X) / P(Y). How SURPRISING the rule is. Lift = 1
#               means X and Y are independent. Lift > 1 is positive
#               association — X raises the probability of Y. Lift < 1 is
#               negative association (substitutes, not complements).
#
#   CONVICTION  conv(X -> Y) = (1 - supp(Y)) / (1 - conf(X -> Y))
#               Directional strength. Unlike lift (symmetric), conviction
#               asymmetrically measures how much X IMPLIES Y. Conviction
#               of 1 = independence; conviction of infinity = perfect
#               implication (conf = 1, never violated).
#
# THE THREE-THRESHOLD FILTER (the only one that matters in practice)
#
#   keep the rule if:
#     support >= s_min        (enough baskets -> statistical power)
#     confidence >= c_min     (reliable enough to act on)
#     lift > 1                (surprising enough to be informative)
#
# Any ONE of these alone is a trap: high-confidence low-lift rules are
# just popularity (everyone buys milk, so any X -> milk has high
# confidence but zero insight). Low-support high-lift rules are often
# just statistical artefacts.



## TASK 2 — BUILD: rule generator + three-threshold filter


Small self-contained Apriori — same contract as 5.1.


For every frequent itemset of size >= 2, every non-empty proper subset
    is a candidate antecedent; its complement is the consequent. We score
    each candidate with support / confidence / lift / conviction and
    drop those below ``min_confidence``.


Apply the three-threshold filter and sort by descending lift.


In [ ]:
def _apriori(
    transactions: list[set[str]], min_support: float
) -> dict[frozenset[str], float]:
    n = len(transactions)
    min_count = min_support * n
    item_counts: dict[str, int] = defaultdict(int)
    for txn in transactions:
        for item in txn:
            item_counts[item] += 1
    freq: dict[frozenset[str], float] = {}
    level: list[frozenset[str]] = []
    for item, count in item_counts.items():
        if count >= min_count:
            fs = frozenset([item])
            freq[fs] = count / n
            level.append(fs)
    k = 2
    while level:
        prev_set = set(level)
        candidates: set[frozenset[str]] = set()
        for i, a in enumerate(level):
            for b in level[i + 1 :]:
                u = a | b
                if len(u) == k and all((u - frozenset([it])) in prev_set for it in u):
                    candidates.add(u)
        if not candidates:
            break
        counts: dict[frozenset[str], int] = defaultdict(int)
        for txn in transactions:
            tf = frozenset(txn)
            for c in candidates:
                if c.issubset(tf):
                    counts[c] += 1
        level = []
        for c, ct in counts.items():
            if ct >= min_count:
                freq[c] = ct / n
                level.append(c)
        k += 1
    return freq


def generate_rules(
    freq_itemsets: dict[frozenset[str], float],
    min_confidence: float,
) -> list[dict]:
    rules: list[dict] = []
    for itemset, support in freq_itemsets.items():
        if len(itemset) < 2:
            continue
        items = list(itemset)
        for r in range(1, len(items)):
            for ant_tuple in combinations(items, r):
                antecedent = frozenset(ant_tuple)
                consequent = itemset - antecedent

                supp_ant = freq_itemsets.get(antecedent)
                supp_con = freq_itemsets.get(consequent)
                if supp_ant is None or supp_con is None:
                    continue

                confidence = support / supp_ant
                if confidence < min_confidence:
                    continue

                lift = confidence / supp_con
                conviction = (
                    (1 - supp_con) / (1 - confidence)
                    if confidence < 1.0
                    else float("inf")
                )

                rules.append(
                    {
                        "antecedent": antecedent,
                        "consequent": consequent,
                        "support": support,
                        "confidence": confidence,
                        "lift": lift,
                        "conviction": conviction,
                    }
                )
    return rules


def filter_actionable(
    rules: list[dict],
    min_support: float,
    min_confidence: float,
    min_lift: float,
) -> list[dict]:
    kept = [
        r
        for r in rules
        if r["support"] >= min_support
        and r["confidence"] >= min_confidence
        and r["lift"] > min_lift
    ]
    kept.sort(key=lambda r: -r["lift"])
    return kept



## TASK 3 — TRAIN: mine + score rules on the SG retail baskets


In [ ]:
transactions = generate_transactions(n=2500, seed=42)
print_transaction_summary(transactions)

MIN_SUPPORT = 0.03
MIN_CONFIDENCE = 0.3

print("\n=== Mining frequent itemsets ===")
frequent = _apriori(transactions, min_support=MIN_SUPPORT)
print(f"  Frequent itemsets: {len(frequent)}")

print("\n=== Generating association rules ===")
rules = generate_rules(frequent, min_confidence=MIN_CONFIDENCE)
print(f"  Rules at min_confidence={MIN_CONFIDENCE}: {len(rules)}")

actionable = filter_actionable(
    rules, min_support=0.03, min_confidence=0.4, min_lift=1.5
)
print(f"  Actionable (supp>=0.03, conf>=0.4, lift>1.5): {len(actionable)}")



### Checkpoint


In [ ]:
assert len(rules) > 0, "At least one rule should clear min_confidence"
for rule in rules[:10]:
    assert 0 <= rule["confidence"] <= 1.0, "confidence must be a probability"
    assert rule["lift"] > 0, "lift must be positive"
assert len(actionable) > 0, "At least one rule should clear the three thresholds"
assert actionable[0]["lift"] > 1.5, "Top actionable rule should have lift > 1.5"
print(
    "\n[ok] Checkpoint passed — rule metrics are valid and actionable set is non-empty\n"
)



## TASK 4 — VISUALISE: top rules + category breakdown + polars scatter


In [ ]:
print("Top 15 actionable rules by lift:")
header = f"  {'Antecedent':<28} {'->':>3} {'Consequent':<20} "
header += f"{'Supp':>6} {'Conf':>6} {'Lift':>6} {'Conv':>7}"
print(header)
print("  " + "-" * 88)
for rule in actionable[:15]:
    ant = format_itemset(rule["antecedent"])
    con = format_itemset(rule["consequent"])
    conv = (
        f"{rule['conviction']:.2f}" if rule["conviction"] != float("inf") else "   inf"
    )
    print(
        f"  {ant:<28} {'->':>3} {con:<20} "
        f"{rule['support']:>6.3f} {rule['confidence']:>6.3f} "
        f"{rule['lift']:>6.2f} {conv:>7}"
    )

# Category breakdown
cross = 0
within = 0
for rule in actionable:
    _, _, rel = categorise_rule(rule["antecedent"], rule["consequent"])
    if rel.startswith("within-category"):
        within += 1
    else:
        cross += 1

print("\n=== Category Breakdown ===")
print(f"  Cross-category rules: {cross}")
print(f"  Within-category rules: {within}")

# Polars frame of every rule (also saved for later lessons)
scatter_df = rules_to_polars(rules).sort("lift", descending=True)
scatter_df.write_csv(OUTPUT_DIR / "top_rules_scatter.csv")
print(f"\n  Saved: {OUTPUT_DIR / 'top_rules_scatter.csv'}")



### Visualisation


In [ ]:
# Support (x) vs confidence (y), coloured by lift. The dashed lines are
# the actionable thresholds: every rule in the top-right box AND with
# lift > 1.5 survives the three-threshold filter. Rules high on the y-axis
# but with lift ~1 are the "popularity" trap from the theory section.
import plotly.graph_objects as go  # noqa: E402

rule_labels = [
    f"{a} -> {c}"
    for a, c in zip(
        scatter_df["antecedent"].to_list(), scatter_df["consequent"].to_list()
    )
]
fig_rules = go.Figure(
    go.Scatter(
        x=scatter_df["support"].to_list(),
        y=scatter_df["confidence"].to_list(),
        mode="markers",
        text=rule_labels,
        marker=dict(
            color=scatter_df["lift"].to_list(),
            colorscale="Viridis",
            colorbar=dict(title="Lift"),
            size=8,
            line=dict(width=0.5, color="white"),
        ),
        hovertemplate="%{text}<br>support=%{x:.3f}<br>confidence=%{y:.3f}"
        "<br>lift=%{marker.color:.2f}<extra></extra>",
    )
)
fig_rules.add_vline(x=0.03, line_dash="dash", line_color="grey")
fig_rules.add_hline(y=0.4, line_dash="dash", line_color="grey")
fig_rules.update_layout(
    title=(
        f"All {len(rules)} rules: support vs confidence, colour = lift "
        f"({len(actionable)} actionable)"
    ),
    xaxis_title="Support",
    yaxis_title="Confidence",
    height=550,
    width=900,
)
rules_path = OUTPUT_DIR / "03_rule_scatter.html"
fig_rules.write_html(str(rules_path))
print(f"[viz] Rule quality scatter: {rules_path}")

# INTERPRETATION: Cross-category rules are the commercially interesting
# ones. Within-category rules (shampoo + soap) mostly restate what a
# store manager already knows — they both sit in the personal-care
# aisle. Cross-category rules (breakfast items -> beverages) are where
# Association Rules pay for themselves: they surface adjacencies you
# did not explicitly plan for.



## TASK 5 — APPLY: health & beauty cart-page recommender


In [ ]:
# SCENARIO: A Singapore health & beauty retailer runs ~100 stores plus
# an online channel. The CRM team wants a "buy-together" recommender that shows
# a shopper exactly ONE extra product on the cart page, chosen to
# maximise incremental basket value. The recommendation MUST be:
#   - Reliable  (high confidence — doesn't annoy shoppers with misses)
#   - Surprising (high lift — not just "you bought shampoo, try soap")
#   - Popular   (high support — enough stock to fulfil)
#
# The three-threshold filter above is literally the product spec for
# this recommender: support gates inventory, confidence gates annoyance,
# lift gates relevance.
#
# BUSINESS IMPACT (illustrative assumptions, not measured figures): take
# an online channel with S$200M/year GMV. If 10% of orders see a
# recommendation and those orders' value rises 3%, that is ~S$600K/year
# of extra sales (S$200M x 10% x 3%); margin is a fraction of that.
# Measure the real conversion and order-value lift with an A/B test
# against a "most popular item" control before scaling.
#
# LIMITATIONS:
#   - Association rules are backward-looking; seasonal or new-launch items
#     need a cold-start fallback (Ex 7 matrix factorisation helps here)
#   - A rule is a correlation, not a preference — always pair with
#     business rules (don't recommend a more expensive substitute to a
#     shopper who just demonstrated price sensitivity)



## TRACK — Log rule-quality metrics to the kailash-ml ExperimentTracker


In [ ]:
# Logs the four-metric distribution across all rules + the actionable cut +
# the cross-vs-within category split. Series = sorted-lift / sorted-conf /
# sorted-support arrays so the M4 dashboard can plot the rule-quality
# distribution alongside the other techniques in this experiment.

import math  # noqa: E402

lifts = [float(r["lift"]) for r in rules]
confs = [float(r["confidence"]) for r in rules]
supps = [float(r["support"]) for r in rules]
finite_convs = [
    float(r["conviction"]) for r in rules if not math.isinf(r["conviction"])
]
mean_finite_conv = sum(finite_convs) / len(finite_convs) if finite_convs else 0.0
n_inf_conv = sum(1 for r in rules if math.isinf(r["conviction"]))

actionable_lifts = [float(r["lift"]) for r in actionable]
top_lift = max(actionable_lifts) if actionable_lifts else 0.0
mean_act_lift = (
    sum(actionable_lifts) / len(actionable_lifts) if actionable_lifts else 0.0
)

track_run(
    tracker,
    exp_name,
    run_name="rule_evaluation_three_threshold",
    params={
        "algorithm": "association_rules",
        "implementation": "from_scratch",
        "n_transactions": len(transactions),
        "min_support_mining": MIN_SUPPORT,
        "min_confidence_filter": MIN_CONFIDENCE,
        "actionable_min_support": 0.03,
        "actionable_min_confidence": 0.4,
        "actionable_min_lift": 1.5,
    },
    scalar_metrics={
        "n_frequent_itemsets": float(len(frequent)),
        "n_rules_generated": float(len(rules)),
        "n_rules_actionable": float(len(actionable)),
        "actionable_rate": float(len(actionable) / max(len(rules), 1)),
        "cross_category_rules": float(cross),
        "within_category_rules": float(within),
        "top_lift": float(top_lift),
        "mean_actionable_lift": float(mean_act_lift),
        "mean_lift_all": float(sum(lifts) / max(len(lifts), 1)),
        "mean_confidence_all": float(sum(confs) / max(len(confs), 1)),
        "mean_support_all": float(sum(supps) / max(len(supps), 1)),
        "n_inf_conviction": float(n_inf_conv),
        "mean_finite_conviction": float(mean_finite_conv),
    },
    series_metrics={
        "lift_sorted_desc": sorted(lifts, reverse=True)[:50],
        "confidence_sorted_desc": sorted(confs, reverse=True)[:50],
        "support_sorted_desc": sorted(supps, reverse=True)[:50],
    },
)
print(f"  [tracked] Rule-quality distribution logged to {exp_name}\n")



## DESTINATION-FIRST CLOSE — mlxtend.frequent_patterns.association_rules


In [ ]:
# This lesson hand-rolled rule generation, all four quality metrics, and
# the three-threshold filter — ~210 lines of structure to internalise WHY
# each metric catches a different failure mode. The production destination
# is the same one you used in lesson 02:
#
#   from mlxtend.frequent_patterns import fpgrowth, association_rules
#
# It runs FP-Growth + every metric (support, confidence, lift, conviction,
# leverage, zhang's metric) in two calls and returns a pandas frame ready
# for the same three-threshold filter you just implemented.

from mlxtend.frequent_patterns import association_rules as mlx_rules  # noqa: E402
from mlxtend.frequent_patterns import fpgrowth as mlx_fpgrowth  # noqa: E402

onehot_pd = transactions_to_onehot(transactions).to_pandas().astype(bool)
mlx_freq = mlx_fpgrowth(onehot_pd, min_support=MIN_SUPPORT, use_colnames=True)
mlx_rules_df = mlx_rules(mlx_freq, metric="confidence", min_threshold=MIN_CONFIDENCE)
mlx_actionable = mlx_rules_df[
    (mlx_rules_df["support"] >= 0.03)
    & (mlx_rules_df["confidence"] >= 0.4)
    & (mlx_rules_df["lift"] > 1.5)
]
print(f"  mlxtend rules : {len(mlx_rules_df)}  hand-rolled : {len(rules)}")
print(
    f"  mlxtend actionable: {len(mlx_actionable)}  "
    f"hand-rolled actionable: {len(actionable)}"
)
print()
print("  Same four metrics, same three-threshold filter — the engine ships")
print("  what you just built. Take the contract; ship the model card.\n")



## REFLECTION


[x] Generated directional association rules from frequent itemsets
  [x] Computed support, confidence, lift, and conviction for every rule
  [x] Applied the three-threshold filter (support + confidence + lift)
  [x] Separated cross-category rules from within-category rules
  [x] Plotted every rule's support vs confidence coloured by lift
  [x] Identified a production scenario (cart-page recommender)
      where the three-threshold filter IS the product spec
  [x] Reproduced the entire pipeline via two mlxtend calls — fpgrowth +
      association_rules — confirming the production destination

  KEY INSIGHT: Confidence alone is popularity. Lift alone is noise.
  Support alone is volume. The three together are actionability.

  Next: 04_rule_features.py — use the discovered rules as features for a
  supervised classifier and measure whether they beat raw product presence.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

# Drain the aiosqlite worker threads so Py_Finalize doesn't hang.
teardown_engines(tracker)

